# 分類に効いている単語を調べる (線形モデルの係数)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/feature_importance.ipynb)

NLP クックブック 12-03 ／ テキスト分類

モデルがどの単語を手がかりに各クラスと判断しているのかを確認したい。データのリーク (カンニング) の発見にも役立つ。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, tarfile, urllib.request
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
tagger = Tagger("-Owakati")
vec = TfidfVectorizer(token_pattern=r"\S+", min_df=3, max_df=0.5, sublinear_tf=True)
X = vec.fit_transform(tagger.parse(t[:2000]).strip() for t in df["text"])
clf = LogisticRegression(C=10, max_iter=1000).fit(X, df["label"])

In [3]:
vocab = vec.get_feature_names_out()
top = {label: vocab[np.argsort(-coef)[:12]] for label, coef in zip(clf.classes_, clf.coef_)}
pd.DataFrame(top)

,dokujo-tsushin,it-life-hack,kaden-channel,livedoor-homme,movie-enter,peachy,smax,sports-watch,topic-news
0,独女,■,話題,livedoor,映画,まし,max,選手,掲示
1,歳,togetter,】,ゴルフ,『,peachy,エス,試合,板
2,女性,見る,【,homme,』,女子,マックス,サッカー,ネット
3,オフィスエムツー,クチコミ,売れ筋,—,本作,さん,s,戦,関連
4,…,：,ビデオ,ビジネス,公開,女性,on,代表,【
5,女,pc,多い,転職,作品,ね,フォン,ファン,】
6,だろう,ニコニコ,salon,クルマ,—,ませ,smaxjp,野球,情報
7,私,製品,パナソニック,求人,movie,2010,スマート,明かし,報じ
8,さん,もっと,家電,シャツ,dvd,ご,執筆,アスリート,記事
9,漫画,it,３,オトコ,enter,でしょう,-,日本,批判


In [4]:
# 1 つの文書について、どの語がどれだけ寄与したか (tf-idf 値 × 係数)
i = 0
row = X[i].toarray().ravel()
c = list(clf.classes_).index(df["label"][i])
contrib = row * clf.coef_[c]
print(df["label"][i], "|", df["title"][i])
print([(vocab[j], round(float(contrib[j]), 2)) for j in np.argsort(-contrib)[:10]])

topic-news | 悪評が次から次へと溢れ出る川越シェフ
[('detail', 0.32), ('関連', 0.29), ('livedoor', 0.26), ('【', 0.26), ('】', 0.26), ('記事', 0.21), ('news', 0.2), ('com', 0.17), ('ネット', 0.16), ('/', 0.15)]


- カテゴリ名そのもの (「独女」「peachy」「smax」) や、本文に含まれる関連記事の URL の断片 (`livedoor`、`detail`) が上位に来る。現実のタスクでは、こうした「答えが漏れている」特徴に頼っていないかを確認する。